# Reference copy - needs non-public data

이 노트북은 원본 연구 노트북의 **참고용 사본**입니다. 실행에 필요한 DataGuide 엑셀 추출본(PER, 재무, 가격 등)은(는) 유료/비공개 데이터라 이 저장소에 포함하지 않았으므로 **그대로는 끝까지 실행되지 않습니다.** 출력, 실행 카운트, 로컬 경로는 제거했습니다. 실행 가능한 데모는 `00_public_sample_workflow.ipynb`(합성 데이터)를 사용하세요.

In [ ]:
import copy
import warnings
warnings.filterwarnings('ignore')

import re
import pandas as pd
import numpy as np
from tqdm import tqdm
import seaborn as sns
import statsmodels.api as sm
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


from scipy.stats.mstats import winsorize

In [ ]:
def winsorize_row(row, lower=0.005, upper=0.005):
    
    q_low = row.quantile(lower)
    q_high = row.quantile(1 - upper)
    
    return row.clip(lower=q_low, upper=q_high)

In [ ]:
idx = pd.IndexSlice

financial_keywords = ['은행','증권','캐피탈','보험','카드','자산','운용','신탁','저축','금고','신용','스팩','금융','화재']


## **Data Load & Preprocessing**

### **1. PER Winsorising**

In [ ]:
PER = pd.read_excel("통합 문서1.xlsx", index_col=0, header=8, sheet_name="PER").iloc[5:]
PER.index = pd.to_datetime(PER.index)
PER.index.name = None

PER = PER.resample("ME").last()

In [ ]:
# ---------------------------------------------------------
# [Step 1] 0을 NaN으로 치환 (ZeroDivisionError 방지)
# ---------------------------------------------------------
PER = PER.replace(0, np.nan)

# ---------------------------------------------------------
# [Step 2] E/P 역수 변환 (Earnings Yield)
# ---------------------------------------------------------
# 적자(음수)는 음수 E/P로, 흑자(양수)는 양수 E/P로 변환됨
EP = 1 / PER

# 혹시 모를 무한대(inf) 처리 (데이터 타입 안정성 확보)
EP = EP.replace([np.inf, -np.inf], np.nan)

# ---------------------------------------------------------
# [Step 3] Cross-sectional Winsorization (1% ~ 99%)
# ---------------------------------------------------------
# 매월(횡단면) 기준으로 상/하위 1% 극단치를 경계값으로 눌러줌

def winsorize_series(s, lower=0.01, upper=0.99):
    # 유효한 데이터가 너무 적으면(예: 5개 미만) winsorize 하지 않음
    if s.count() < 5:
        return s
    
    down = s.quantile(lower)
    up = s.quantile(upper)
    return s.clip(lower=down, upper=up)

# axis=1 (행 방향, 즉 매월 횡단면 적용)
print("Winsorizing data... (This may take a while)")
EP_winsor = EP.apply(lambda row: winsorize_series(row, 0.01, 0.99), axis=1)

# ---------------------------------------------------------
# [Step 4] 불필요한 컬럼 제거 (Financial keywords 포함된 것들)
# ---------------------------------------------------------
EP_winsor = EP_winsor.drop(columns=EP_winsor.filter(regex='|'.join(financial_keywords)).columns)


In [ ]:
print("===== [E/P Transformation Complete] =====")
print("Data Shape:", EP_winsor.shape)
print("Sample Data (First 5 rows):")
print(EP_winsor.iloc[:5, :5]) # 5개 종목만 확인

In [ ]:
PER_winsor = EP_winsor.copy()

**[Data Preprocessing Report] Target Variable Construction ($E/P$)**

**1. 데이터 로드 및 시계열 정렬 (Data Acquisition & Alignment)**
* **Source:** `통합 문서1.xlsx` 내 `PER` 시트로부터 Raw Data 추출.
* **Frequency:** 월말(Month-End) 기준 리샘플링(`resample("ME").last()`)을 통해 시계열 정합성 확보.

**2. 결측치 및 에러 처리 (Error Handling)**
* **Zero Replacement:** PER이 0인 관측치는 경제적 해석이 불가능(Price=0 or Error)하며 역수 변환 시 `ZeroDivisionError`를 유발하므로 `NaN`으로 치환하여 제거.

**3. 변수 변환 (Variable Transformation: $P/E \rightarrow E/P$)**
* **Inverse Transformation:** 기존 $P/E$ 대신 **$E/P$ (Earnings Yield)**를 사용.
* **Rationale:**
    * **적자 기업 포함:** 적자(Negative Earnings) 발생 시 $P/E$의 불연속성 문제를 해결하고, 음수($-$) 수익률로 연속성을 유지하며 모델에 포함.
    * **선형성 확보:** 가격(Price)이 분모로 이동함에 따라, 밸류에이션 지표의 분포를 보다 선형적(Linear)이고 대칭적(Symmetric)으로 개선.

**4. 이상치 제어 (Outlier Control)**
* **Method:** **Cross-sectional Winsorization**.
* **Threshold:** 매월 횡단면 데이터 기준 **하위 1%** 및 **상위 1%** ($1^{st} \sim 99^{th}$ percentile)에서 극단값을 경계값으로 대체(Clipping).
* **Purpose:** 데이터의 손실 없이 비정상적인 극단치(Extreme Outliers)가 회귀분석 및 ML 학습에 미치는 왜곡 최소화.

**5. 섹터 필터링 (Sector Filtering)**
* **Exclusion:** 금융업 제외.

---
### **2. 설명 변수 전처리**
각 카테고리별 재무 지표를 생성합니다.
* 주의: 이 단계에서는 시차(Lag)를 적용하지 않고, 원본 데이터를 계산만 합니다.
* 추후 분석 단계에서 Price가 포함된 변수(M/B, Beta 등)를 제거할 수 있도록 모두 생성합니다.

In [ ]:
# ---------------------------------------------------------
# 1. 데이터 로딩 유틸리티 함수 (Code Refactoring)
# ---------------------------------------------------------
def load_quantiwise_sheet(file_path, sheet_name):
    """
    Quantiwise 양식의 엑셀 시트를 로드하고 전처리하는 함수
    - Header: 8행(Ticker), 13행(Item) -> MultiIndex Columns
    - Index: Date (0열)
    - Resample: Month End (ME)
    """
    print(f"Loading '{sheet_name}'...")
    try:
        # header=[8, 13]은 0-based index 기준 8번째, 13번째 행을 헤더로 씀
        df = pd.read_excel(file_path, index_col=0, header=[8, 13], sheet_name=sheet_name)
        
        df.index = pd.to_datetime(df.index)
        df.index.name = "Date"
        
        # 월말 리샘플링 (Last observation)
        return df.resample("ME").last()
        
    except Exception as e:
        print(f"Error loading {sheet_name}: {e}")
        return pd.DataFrame()

In [ ]:
# ---------------------------------------------------------
# 2. Raw Data 로드 실행
# ---------------------------------------------------------
FILE_PATH = "통합 문서1.xlsx"  # 파일명은 자네 환경에 맞게 수정하게

quality_raw = load_quantiwise_sheet(FILE_PATH, "Quality")
growth_raw1 = load_quantiwise_sheet(FILE_PATH, "Growth1")
growth_raw2 = load_quantiwise_sheet(FILE_PATH, "Growth2")
inveset_raw = load_quantiwise_sheet(FILE_PATH, "Investment")
risk_raw    = load_quantiwise_sheet(FILE_PATH, "Risk")
etc_raw     = load_quantiwise_sheet(FILE_PATH, "etc")

print("\nAll Raw Data Loaded Successfully.")

In [ ]:
# 공통 Ticker 추출 (모든 데이터셋에 존재하는 종목만 분석)
# 1. 공통된 Ticker 리스트 확보
common_tickers = list(
    set(growth_raw1.columns.get_level_values(0)) & 
    set(quality_raw.columns.get_level_values(0)) & 
    set(inveset_raw.columns.get_level_values(0)) & 
    set(risk_raw.columns.get_level_values(0)) & 
    set(etc_raw.columns.get_level_values(0)) &
    set(PER_winsor.columns) # Target이 있는 종목만
)

print(f"Number of Common Tickers: {len(common_tickers)}")

In [ ]:
for name in tqdm(common_tickers, desc="Feature Eng"):
    # 1. Quality
    # 발생액
    quality_raw[(name, "Accruals")] = (quality_raw[(name, "당기순이익(TTM)")] - quality_raw[(name, "영업활동으로인한현금흐름(TTM)")]) / growth_raw1[(name, "자산총계")]
    # EPS 변동성
    quality_raw[(name, "EPS_Vol")] = np.log1p(quality_raw[(name, "EPS(지배, Adj., TTM)")].rolling(36).std())

    # 2. Growth
    # 매출액 성장률
    growth_raw1[(name, "Sales_Growth")] = np.log1p(growth_raw1[(name, "매출액(TTM)")]) - np.log1p(growth_raw1[(name, "매출액(TTM)")].shift(12))
    # 투자의 적극성
    growth_raw1[(name, "CAPEX_Asset")] = growth_raw1[(name, "CAPEX")] / growth_raw1[(name, "자산총계")]
    # 무형자산 비중
    growth_raw1[(name, "Intangible_Asset")] = growth_raw1[(name, "무형자산")] / growth_raw1[(name, "자산총계")]
    # [Price Var] M/B
    growth_raw1[(name, "M_B")] = growth_raw2[(name, "시가총액")] / growth_raw2[(name, "자본총계")]
    # [Pure Fundamental] Size (Log Assets)
    growth_raw1[(name, "Log_Asset")] = np.log(growth_raw1[(name, "자산총계")])

    # 3. Investment
    # 자사주 매입 강도
    inveset_raw[(name, "Buyback_Yield")] = inveset_raw[(name, "보통주기중취득자기주식수")] / inveset_raw[(name, "보통주평균발행주식수")]
    
    # 4. Etc (Liquidity)
    etc_raw[(name, "Cash_Ratio")] = etc_raw[(name, "현금및현금성자산")] / growth_raw1[(name, "자산총계")]

In [ ]:
dataset_list = []

for name in tqdm(common_tickers, desc="Building Panel"):
    # -------------------------------------------------------
    # (1) 해당 종목의 모든 Feature 모으기 (DataFrame 생성)
    # -------------------------------------------------------
    tmp = pd.DataFrame(index=PER_winsor.index) # 시계열 인덱스 기준
    
    # --- Target (Lag 적용 X) ---
    tmp['EP'] = PER_winsor[name]  # 앞서 전처리한 E/P (Earnings Yield)
    
    # --- Features (Lag 적용 대상) ---
    # Quality
    tmp['ROE'] = quality_raw[(name, "ROE(지배, TTM)")]
    tmp['Accruals'] = quality_raw[(name, "Accruals")]
    tmp['EPS_Vol'] = quality_raw[(name, "EPS_Vol")]
    
    # Growth
    tmp['Sales_Growth'] = growth_raw1[(name, "Sales_Growth")]
    tmp['CAPEX_Asset'] = growth_raw1[(name, "CAPEX_Asset")]
    tmp['Intangible_Asset'] = growth_raw1[(name, "Intangible_Asset")]
    tmp['M_B'] = growth_raw1[(name, "M_B")] # [Price Var]
    
    # Investment
    tmp['Payout_Ratio'] = inveset_raw[(name, "배당성향(%)")].shift(6)
    tmp['Asset_Turnover'] = inveset_raw[(name, "총자산회전율")]
    tmp['Buyback_Yield'] = inveset_raw[(name, "Buyback_Yield")]
    
    # Risk
    tmp['Debt_Ratio'] = risk_raw[(name, "부채비율")]
    tmp['Interest_Cov'] = risk_raw[(name, "이자보상배율")]
    tmp['Beta'] = risk_raw[(name, "60개월베타")] # [Price Var]
    
    # Liquidity
    tmp['Cash_Ratio'] = etc_raw[(name, "Cash_Ratio")]
    
    # Size
    tmp['Log_Asset'] = growth_raw1[(name, "Log_Asset")] # [Pure Fundamental]
    tmp['Log_MarketCap'] = np.log(growth_raw2[(name, "시가총액")]).shift(1) # [Price Var]
    
    # -------------------------------------------------------
    # (2) Lag 적용 (6개월) - Target을 제외한 모든 변수
    # -------------------------------------------------------
    features = tmp.columns.drop(['EP'])
    tmp[features] = tmp[features].shift(6)
    
    # -------------------------------------------------------
    # (3) 메타 데이터 추가 (Ticker)
    # -------------------------------------------------------
    tmp['Ticker'] = name
    tmp['Date'] = tmp.index # Date 컬럼 명시화
    
    # 결측치 제거 (Target이 없거나, Lag로 인해 앞부분 빈 데이터 제거)
    tmp.dropna(inplace=True)
    
    dataset_list.append(tmp)

# 하나의 거대한 DataFrame으로 합치기
full_data = pd.concat(dataset_list, axis=0, ignore_index=True)

# 인덱스 재설정 (MultiIndex: Date, Ticker) -> 분석하기 편함
full_data.set_index(['Date', 'Ticker'], inplace=True)
full_data.sort_index(inplace=True)

print(f"Final Dataset Shape: {full_data.shape}")
full_data.head()

---
# **데이터 저장**

In [ ]:
# 1. CSV로 저장 (인덱스인 Date, Ticker를 포함하여 저장)
full_data.to_csv("Research_Data_Final.csv", encoding="CP949")
print("Data saved to 'Research_Data_Final.csv'")